# 🔨 Forge Quickstart — Tune a Small Model on Colab T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aevyra/forge/blob/main/notebooks/forge_quickstart.ipynb)

**What this does:** runs Forge's autonomous vLLM config optimizer on
`Qwen/Qwen3-8B-AWQ` (fits in a free T4's 16 GB VRAM).
The LLM agent proposes config mutations; Forge keeps or reverts each one
based on measured throughput.

**Runtime requirements**
- *Runtime → Change runtime type → T4 GPU*
- OpenRouter API key **or** use the dry-run section (no key needed)
- Estimated time: 20–40 min for a 10-experiment real run

**Sections**
1. [Setup](#scrollTo=setup)
2. [🧪 Dry-run demo](#scrollTo=dryrun) — no vLLM, no API key needed
3. [🚀 Real run](#scrollTo=realrun) — live vLLM + LLM agent
4. [📊 Results & download](#scrollTo=results)


In [ ]:
# @title ① Check GPU
import subprocess

result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    raise RuntimeError(
        "No GPU detected.\nGo to  Runtime → Change runtime type → T4 GPU  and try again."
    )
print("GPU:", result.stdout.strip())

In [ ]:
# @title ② Install dependencies
%pip install -q aevyra-forge[notebooks] \
    "vllm>=0.4,<0.7" \
    "openai>=1.0" \
    "httpx>=0.27" \
    "typer>=0.12"

print("✓ Install complete")

In [ ]:
# @title ③ Imports
import os
import logging
import shutil
from pathlib import Path
from IPython.display import display
import pandas as pd

from aevyra_forge.recipe import HardwareSpec
from aevyra_forge.workload import workload_synthetic, workload_shared_prefix
from aevyra_forge.playbook import Playbook
from aevyra_forge.orchestrator import Orchestrator, ForgeConfig
from aevyra_forge.result import ForgeStore

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-8s %(message)s",
    datefmt="%H:%M:%S",
)
print("✓ Imports OK")

---
<a name="dryrun"></a>
## 🧪 Section 1 — Dry-run demo

Runs the full orchestrator loop with **synthetic bench results** — no vLLM
process starts, no API key is needed.  Good for understanding the loop before
spending GPU time or credits.


In [ ]:
# @title ④ Dry-run config
# @markdown How many experiments to run in dry-run mode
DRY_RUN_EXPERIMENTS = 8  # @param {type:"slider", min:3, max:20, step:1}

hardware = HardwareSpec(
    vendor="nvidia",
    gpu_type="T4",
    count=1,
    memory_gb_per_gpu=16,
)

# Chat-shaped synthetic workload (short inputs, medium outputs)
workload = workload_synthetic(
    n_requests=200,
    avg_input_tokens=256,
    avg_output_tokens=128,
    duration_s=60.0,
    seed=42,
)
print(workload.summary())

In [ ]:
# @title ⑤ Stub agent (no API key needed)
# Stub LLM: cycles through sensible T4 mutations so you can see
# the keep/revert loop without spending API credits.
import json as _json

_DRY_MUTATIONS = [
    {
        "layer": "config",
        "changes": {"max_num_seqs": 64},
        "rationale": "T4 baseline uses 32; 64 increases batch utilization on this GPU.",
    },
    {
        "layer": "config",
        "changes": {"enable_prefix_caching": True},
        "rationale": "Chat workloads share system-prompt prefixes — caching reduces TTFT.",
    },
    {
        "layer": "config",
        "changes": {"max_num_batched_tokens": 4096},
        "rationale": "Doubling batched tokens lets vLLM fill the GPU on mixed-length batches.",
    },
    {
        "layer": "config",
        "changes": {"gpu_memory_utilization": 0.92},
        "rationale": "Default 0.88 is conservative; 0.92 reclaims ~600 MB for the KV cache.",
    },
    {
        "layer": "config",
        "changes": {"enable_chunked_prefill": True},
        "rationale": "Chunked prefill hides prefill spikes under decode on memory-limited GPUs.",
    },
    {
        "layer": "config",
        "changes": {"max_num_seqs": 32},
        "rationale": "64-seq batch stressed VRAM; stepping back to 32 to confirm stability.",
    },
    {
        "layer": "config",
        "changes": {"block_size": 16},
        "rationale": "Smaller block size reduces fragmentation on 16 GB VRAM.",
    },
    {
        "layer": "config",
        "changes": {},
        "rationale": "No further improvement expected — search converged.",
    },
]

_mutation_iter = iter(_DRY_MUTATIONS)


def stub_agent(prompt: str) -> str:
    mut = next(_mutation_iter, _DRY_MUTATIONS[-1])
    return _json.dumps(
        {
            "rationale": mut["rationale"],
            "mutation": {"layer": mut["layer"], "changes": mut["changes"]},
            "expected_throughput_delta_pct": 5.0,
            "expected_accuracy_delta": 0.0,
        }
    )


stub_agent.tokens_used = 0
print("✓ Stub agent ready")

In [ ]:
# @title ⑥ Run dry-run orchestrator
run_dir = Path("/tmp/forge_dryrun")
if run_dir.exists():
    shutil.rmtree(run_dir)
run_dir.mkdir(parents=True)

_mutation_iter = iter(_DRY_MUTATIONS)  # reset

store = ForgeStore(run_dir)
playbook = Playbook(raw_markdown="")

forge_cfg = ForgeConfig(
    max_experiments=DRY_RUN_EXPERIMENTS,
    max_wall_clock_hours=1.0,
    dry_run=True,
    convergence_window=4,
    min_improvement_pct=1.0,
)

orc = Orchestrator(
    model="Qwen/Qwen3-8B-AWQ",
    hardware=hardware,
    workload=workload,
    playbook=playbook,
    llm=stub_agent,
    store=store,
    forge_config=forge_cfg,
)

best_recipe, history = orc.run()
print(f"\n✓ Finished — {len(history)} experiments")

In [ ]:
# @title ⑦ Dry-run results
def _results_df(history):
    rows = []
    for e in history:
        br = e.bench_result
        rows.append(
            {
                "id": e.id,
                "gen": e.recipe.generation,
                "score": round(e.score, 1) if e.score is not None else None,
                "throughput tok/s": round(br.throughput_tokens_per_sec, 1) if br else None,
                "p99 ms": round(br.p99_latency_ms) if br else None,
                "kept": "✓" if e.kept else "✗",
                "llm tok": e.llm_tokens if e.llm_tokens else "-",
                "rationale": (e.agent_decision.rationale[:65] if e.agent_decision else "baseline"),
            }
        )
    return pd.DataFrame(rows)


df = _results_df(history)
display(
    df.style.map(
        lambda v: (
            "color: green; font-weight: bold"
            if v == "✓"
            else ("color: #cc0000" if v == "✗" else "")
        ),
        subset=["kept"],
    )
)

best = store.latest_run().best()
if best:
    print(f"🏆 Best recipe  gen={best.recipe.generation}  score={best.score:.1f} tok/s")
    print()
    print(best.recipe.to_yaml())

---
<a name="realrun"></a>
## 🚀 Section 2 — Real run (live vLLM + LLM agent)

This section:
1. Downloads `Qwen/Qwen3-8B-AWQ` weights (~3 GB)
2. Boots a vLLM server as a subprocess per experiment
3. Replays the synthetic workload against it to measure real throughput
4. Uses Qwen3-8B via OpenRouter to drive the agent (free tier available)

**Prerequisites:** set your OpenRouter API key below.
Get one free at [openrouter.ai](https://openrouter.ai). Store it in **Colab Secrets** (🔑 icon in the sidebar) as `OPENROUTER_API_KEY`.


In [ ]:
# @title ⑧ Configuration
# @markdown ### OpenRouter API key
# @markdown Get one free at openrouter.ai — store in Colab Secrets as OPENROUTER_API_KEY

try:
    from google.colab import userdata

    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY") or ""
except Exception:
    OPENROUTER_API_KEY = ""

OPENROUTER_API_KEY_OVERRIDE = ""  # @param {type:"string"}
if OPENROUTER_API_KEY_OVERRIDE.strip():
    OPENROUTER_API_KEY = OPENROUTER_API_KEY_OVERRIDE.strip()

# @markdown ---
# @markdown ### Model + budget
MODEL = "Qwen/Qwen3-8B-AWQ"  # @param {type:"string"}
# GPU tiers:
#   T4  (16 GB, free)  → "Qwen/Qwen3-8B-AWQ"        INT4, ~4.5 GB weights
#   L4  (24 GB)        → "Qwen/Qwen3-8B"             BF16, ~16 GB weights
#   A100 (80 GB)       → "Qwen/Qwen3-8B"             BF16, plenty of KV cache
#   G4 RTX Pro 6000    → "Qwen/Qwen3-8B" or 32B+     96 GB, highest throughput
# (Select runtime: Runtime → Change runtime type → T4 / A100 / G4 GPU)

MAX_EXPERIMENTS = 10  # @param {type:"slider", min:5, max:30, step:1}
MAX_HOURS = 2.0  # @param {type:"number"}

if not OPENROUTER_API_KEY:
    print("⚠️  No API key found.")
    print("   Add it via the 🔑 Colab Secrets sidebar (key: OPENROUTER_API_KEY)")
    print("   or paste it into OPENROUTER_API_KEY_OVERRIDE above.")
else:
    os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
    masked = OPENROUTER_API_KEY[:8] + "..." + OPENROUTER_API_KEY[-4:]
    print(f"✓ API key: {masked}")

print(f"Model: {MODEL}  |  max experiments: {MAX_EXPERIMENTS}  |  max hours: {MAX_HOURS}")

In [ ]:
# @title ⑨ Load playbook
import aevyra_forge
from aevyra_forge.playbook import load_playbook

# The bundled playbook is next to the package source; fall back gracefully.
_candidates = [
    Path(aevyra_forge.__file__).parent / "playbook.md",
    Path("/content/forge/playbooks/default.md"),
]
playbook_real = None
for _p in _candidates:
    if _p.exists():
        playbook_real = load_playbook(_p)
        print(f"✓ Playbook loaded from {_p}  ({len(playbook_real.sections)} sections)")
        break

if playbook_real is None:
    playbook_real = Playbook(raw_markdown="")
    print("⚠️  No playbook found — agent will improvise (still works, slightly less efficient)")

In [ ]:
# @title ⑩ Run Forge (real vLLM) — ~3–5 min per experiment
# vLLM stays running between experiments and only restarts when config args change.
assert OPENROUTER_API_KEY, "Set your OpenRouter API key in cell ⑧ first, then re-run that cell."

from aevyra_forge.llm import resolve_llm  # noqa: E402

real_run_dir = Path("/content/forge_real")
if real_run_dir.exists():
    shutil.rmtree(real_run_dir)
real_run_dir.mkdir(parents=True)

hardware_real = HardwareSpec(
    vendor="nvidia",
    gpu_type="T4",
    count=1,
    memory_gb_per_gpu=16,
)

# Shared-prefix workload: every request shares a 512-token system prompt.
# enable_prefix_caching=True (which the agent will discover) skips recomputing
# those tokens after the first request → 20-40% throughput gain on T4.
workload_real = workload_shared_prefix(
    n_requests=100,
    concurrency=8,  # 8 in-flight — makes max_num_seqs tunable
    prefix_tokens=512,
    query_tokens=64,
    output_tokens=128,
    duration_s=120.0,
    seed=42,
)

# Qwen3-8B on OpenRouter — free tier available, no Anthropic key needed
llm = resolve_llm("openrouter/qwen/qwen3-8b")

forge_cfg_real = ForgeConfig(
    max_experiments=MAX_EXPERIMENTS,
    max_wall_clock_hours=MAX_HOURS,
    dry_run=False,
    convergence_window=4,
    min_improvement_pct=1.0,
    accuracy_floor=0.99,
)

orc_real = Orchestrator(
    model=MODEL,
    hardware=hardware_real,
    workload=workload_real,
    playbook=playbook_real,
    llm=llm,
    store=ForgeStore(real_run_dir),
    forge_config=forge_cfg_real,
)

print(f"Starting Forge on {MODEL}...")
print(f"Artifacts → {real_run_dir}")
print()

best_real, history_real = orc_real.run()
print(f"\n✓ Done — {len(history_real)} experiments")

In [ ]:
# @title ⑪ Real-run results
df_real = _results_df(history_real)
display(
    df_real.style.map(
        lambda v: (
            "color: green; font-weight: bold"
            if v == "✓"
            else ("color: #cc0000" if v == "✗" else "")
        ),
        subset=["kept"],
    )
)

best_r = ForgeStore(real_run_dir).best()
if best_r:
    baseline_tps = history_real[0].bench_result.throughput_tokens_per_sec
    best_tps = best_r.bench_result.throughput_tokens_per_sec
    gain_pct = (best_tps - baseline_tps) / baseline_tps * 100

    print(f"\n🏆  Best recipe  gen={best_r.recipe.generation}")
    print(f"    Throughput : {best_tps:.1f} tok/s  (baseline {baseline_tps:.1f}, +{gain_pct:.1f}%)")
    print(f"    P99 latency: {best_r.bench_result.p99_latency_ms:.0f} ms")
    print()
    print(best_r.recipe.to_yaml())

In [ ]:
# @title ⑫ Download best_recipe.yaml
from google.colab import files

recipe_path = real_run_dir / "best_recipe.yaml"
if recipe_path.exists():
    files.download(str(recipe_path))
    print("✓ Downloading best_recipe.yaml")
    print()
    print("Pass this file to your vLLM deployment:")
    print("  forge report /content/forge_real   # view the full TSV")
else:
    print("Run the real-run section first.")

---
## Next steps

| What | How |
|------|-----|
| **Bring your own workload** | Export from Langfuse/OTel → JSONL, then `workload_from_jsonl("trace.jsonl")` |
| **Overnight run** | Set `MAX_EXPERIMENTS=50`, `MAX_HOURS=12` and let it run |
| **AMD MI300X** | `HardwareSpec(vendor="amd", gpu_type="MI300X", count=1, memory_gb_per_gpu=192)` |
| **Custom playbook** | Edit `playbooks/default.md` and pass `--playbook` path |
| **Resume a crashed run** | `forge resume /content/forge_real` |
| **Docs** | [github.com/aevyra/forge](https://github.com/aevyra/forge) |
